# Экспериментальное исследование влияния параметров алгоритмов хранения паролей на вычислительные затраты

## Импорт библиотек

In [13]:
import time
import os
import hashlib
import warnings

import bcrypt
from argon2 import PasswordHasher
from argon2.low_level import hash_secret_raw, Type
import psutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
warnings.filterwarnings('ignore')

%matplotlib inline 

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [ ]:
passwords = []

with open("input_data/passwords.txt", 'r', encoding='utf-8') as f:
    for pwd in f:
        passwords.append(pwd[:-1])

    

["b'odJFCrn*l2edlBD#dz'", "b'*1C5Jau2RJtBRnlWmTS'", "b'Hf6&pWkLUyifDLkDm'", "b'J6UuVTAIjv&Fu7'", "b'ICPhDeOZIiBOB@'", "b'6sHrF&H2ZUCr$@'", "b'gotu2iXW7'", "b'Gbo&IRoL3u6aH#w#'", "b'M#ztVu*^a'", "b'P!coUNEhEkk!i&qq8'", "b'vH^2B*zNZV4%5pFC'", "b'RcDCajhDi'", "b'Qj$EJ!B*'", "b'8F80ymm3T2'", "b'7gmhZRnFyy&5r2'", "b'J7Fj4mg*bl'", "b'Ev0!9BZhvWaXH6K2!tyL'", "b'h*hOhg9#^uh'", "b'kxiiEZpFfk1%OHAO'", "b'HYqM6Ojb6mj'", "b'B#HqSiFVKu4*M^bM'", "b'nrHontIKARAH#!Ggl2'", "b'faQqHu42bojt'", "b'eVs3qfNUfTAFnT0t'", "b'uw0dwQ0FIun'", "b'e8Cz6SNDCdyZQJ'", "b'JS$Z&QdoH'", "b'Hen3SO3$oX'", "b'yGf3a%&zU3iQOpM#N'", "b'0PZLqy1WwMZaMKA3P7'", "b'4B$8vkK$QlENCzs'", "b'fF8j61yX'", "b'ZFsan2Cw%7gFp6r'", "b'7^O4#25u85HFJ%!EJ4jK'", "b'IQO*krtDXtB'", "b'10Q*71hA1'", "b'cW9aTMX1&*C!CI'", "b'!dXRZv7q&dYdk2'", "b'7xgHWPB6PR'", "b'WJ1Gk8c*gSCifdFzctEq'", "b'oB7GVvouNndNWYz'", "b'FnMpfS&2V'", "b'#Rb1!n3U6'", "b't3w%I&973IPFlJ5F7WR'", "b'@Px!BTHR'", "b'Jb%ykE0!E8!5'", "b'clLCZFNV8^S2QT6INGDp'", "b'yOp&xyB9J^KmyLDUwMb'", "b'

## Функция замеров

In [7]:
def measure_memory_and_time(process, func, *args, **kwargs):
    mem_before = process.memory_info().rss / (1024 * 1024)

    start_time = time.perf_counter()
    result = func(*args, **kwargs)
    end_time = time.perf_counter()

    mem_after = process.memory_info().rss / (1024 * 1024)

    time_ms = (end_time - start_time) * 1000
    memory_mb = max(0, mem_after - mem_before)

    return result, time_ms, memory_mb

## Эксперимент

In [10]:
def hash_bcrypt(password, cost_factor):
    salt = bcrypt.gensalt(rounds=cost_factor)
    return bcrypt.hashpw(password.encode('utf-8'), salt)

In [11]:
def run_bcrypt_experiment(passwords, cost_factors, repeats=100):
    results = []

    process = psutil.Process(os.getpid())

    for cost in cost_factors:
        print(f"Тестируем bcrypt с cost={cost}...")

        times = []
        memories = []

        for i in range(repeats):

            pwd = passwords[i % len(passwords)]

            res, time_res, mem_res = measure_memory_and_time(process, hash_bcrypt, pwd, cost)

            times.append(time_res)
            memories.append(mem_res)

        median_time = np.median(times)
        max_memory = max(memories)

        results.append({
            'algorithm': 'bcrypt',
            'parameter_name': 'cost_factor',
            'parameter_value': cost,
            'time_ms': round(median_time, 3),
            'memory_mb': round(max_memory, 3)
        })

    return results

In [14]:
cost_factors_to_test = [4, 6, 8, 10, 12]

bcrypt_results = run_bcrypt_experiment(passwords, cost_factors_to_test, repeats=100)

Тестируем bcrypt с cost=4...
Тестируем bcrypt с cost=6...
Тестируем bcrypt с cost=8...
Тестируем bcrypt с cost=10...
Тестируем bcrypt с cost=12...


In [15]:
df_bcrypt = pd.DataFrame(bcrypt_results)
df_bcrypt.to_csv('results/bcrypt_results.csv', index=False)
df_bcrypt

,algorithm,parameter_name,parameter_value,time_ms,memory_mb
0,bcrypt,cost_factor,4,0.749,0
1,bcrypt,cost_factor,6,2.911,0
2,bcrypt,cost_factor,8,10.985,0
3,bcrypt,cost_factor,10,41.847,0
4,bcrypt,cost_factor,12,167.693,0
